In [ ]:
# ── Install (run once, then restart kernel) ───────────────────────────────────
import sys
!{sys.executable} -m pip install pandas numpy scikit-learn joblib matplotlib seaborn -q
print('Libraries installed ✓')

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

os.makedirs('../outputs', exist_ok=True)
os.makedirs('../models', exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE    = 0.20

print('Imports complete ✓')

In [ ]:
# ── Load raw data ─────────────────────────────────────────────────────────────
RAW_PATH = '../../data/WA_Fn-UseC_-Telco-Customer-Churn.csv'
df = pd.read_csv(RAW_PATH)
print(f'Raw shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head(3)

In [ ]:
# ── Fix TotalCharges ──────────────────────────────────────────────────────────
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(0, inplace=True)

# Drop customerID — not predictive
df.drop(columns=['customerID'], inplace=True)

print(f'After drop: {df.shape[1]} columns')

In [ ]:
# ── Encode target variable ────────────────────────────────────────────────────
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
print(f'Churn distribution:\n{df["Churn"].value_counts()}')
print(f'Churn rate: {df["Churn"].mean()*100:.1f}%')

In [ ]:
# ── Identify column types ─────────────────────────────────────────────────────
TARGET     = 'Churn'
NUM_COLS   = ['tenure', 'MonthlyCharges', 'TotalCharges']
BIN_COLS   = ['SeniorCitizen']  # already 0/1
CAT_COLS   = [
    c for c in df.columns
    if c not in NUM_COLS + BIN_COLS + [TARGET]
    and df[c].dtype == 'object'
]

print(f'Numerical : {NUM_COLS}')
print(f'Binary    : {BIN_COLS}')
print(f'Categorical ({len(CAT_COLS)}): {CAT_COLS}')

In [ ]:
# ── One-hot encode categorical columns ───────────────────────────────────────
# drop_first=True avoids multicollinearity (dummy variable trap)
df_encoded = pd.get_dummies(df, columns=CAT_COLS, drop_first=True)
print(f'Shape after encoding: {df_encoded.shape[0]:,} rows × {df_encoded.shape[1]} columns')
df_encoded.head(2)

In [ ]:
# ── Split features and target ─────────────────────────────────────────────────
X = df_encoded.drop(columns=[TARGET])
y = df_encoded[TARGET]

print(f'Features (X): {X.shape}   |   Target (y): {y.shape}')
print(f'Feature names ({len(X.columns)}):')  
for i, col in enumerate(X.columns, 1):
    print(f'  {i:>2}. {col}')

In [ ]:
# ── Train / test split ────────────────────────────────────────────────────────
# stratify=y ensures churn ratio is preserved in both splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f'Train set : {X_train.shape[0]:,} rows  |  Churn rate: {y_train.mean()*100:.1f}%')
print(f'Test set  : {X_test.shape[0]:,} rows   |  Churn rate: {y_test.mean()*100:.1f}%')
print('Churn rates are balanced across splits ✓')

In [ ]:
# ── Scale numerical features ──────────────────────────────────────────────────
# Fit ONLY on train, transform both — avoids data leakage
scaler = StandardScaler()

X_train[NUM_COLS] = scaler.fit_transform(X_train[NUM_COLS])
X_test[NUM_COLS]  = scaler.transform(X_test[NUM_COLS])

print('Scaling applied (StandardScaler fit on train only) ✓')
print(f'\ntenure mean after scaling: {X_train["tenure"].mean():.4f}  (should be ≈ 0)')
print(f'tenure std  after scaling: {X_train["tenure"].std():.4f}   (should be ≈ 1)')

In [ ]:
# ── Save processed arrays & scaler ───────────────────────────────────────────
joblib.dump(X_train, '../outputs/X_train.pkl')
joblib.dump(X_test,  '../outputs/X_test.pkl')
joblib.dump(y_train, '../outputs/y_train.pkl')
joblib.dump(y_test,  '../outputs/y_test.pkl')
joblib.dump(scaler,  '../models/scaler.pkl')

# Also save feature names for later use
feature_names = list(X.columns)
joblib.dump(feature_names, '../outputs/feature_names.pkl')

print('Saved to predictive-modeling/outputs/ :')
print('  X_train.pkl  X_test.pkl  y_train.pkl  y_test.pkl  feature_names.pkl')
print('Saved to predictive-modeling/models/ :')
print('  scaler.pkl')
print('\nNext → open 02_model_training.ipynb')